# AIRA Model Fine-Tuning — QLoRA SFT on Google Colab
### Autonomous Infrastructure Resilience Architecture

This notebook implements the **Phase 4** SFT (Supervised Fine-Tuning) pipeline for **AIRA**. 
It uses standard **HuggingFace TRL + bitsandbytes QLoRA** training on a free-tier Google Colab T4 GPU to fine-tune `google/gemma-4-e4b-it` on our self-generated adversarial trajectory dataset (`sft_dataset.jsonl`).

At the end of training, it exports the fine-tuned LoRA adapter weights.

### 1. Install Unsloth and Dependencies

In [ ]:
%%capture
# Install standard HuggingFace SFT and bitsandbytes dependencies with transformers from source
!pip install git+https://github.com/huggingface/transformers.git
!pip install trl peft accelerate bitsandbytes datasets
!pip install pydantic structlog

### 2. Load Model and Tokenizer (4-bit Quantization)

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Disable W&B logging, restrict to GPU 0, and configure memory optimizations
os.environ["WANDB_DISABLED"] = "true"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    os.environ["HF_HOME"] = "/kaggle/working/cache"

model_id = "google/gemma-4-e4b-it"
max_seq_length = 512          # Set memory-safe sequence limit to cut logits size in half and avoid OOM

print(f"[*] Loading model and tokenizer in 4-bit QLoRA for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.model_max_length = max_seq_length

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",   # Force PyTorch Scaled Dot Product Attention for memory efficiency
    device_map={"": 0}           # Load completely on GPU 0 to prevent meta splits and speed up training
)
print("[SUCCESS] 4-bit QLoRA Model loaded successfully on GPU!")

### 3. Configure LoRA Adapters (Rank 64 / Alpha 128)

In [ ]:
from peft import LoraConfig, get_peft_model
import torch

peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=".*language_model.*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Convert all remaining bfloat16 parameters and buffers to float16 to prevent GradScaler crash on T4 GPU
print("Casting model parameters and buffers to float16 for T4 compatibility...")
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
for buf in model.buffers():
    if buf.dtype == torch.bfloat16:
        buf.data = buf.data.to(torch.float16)

### 4. Load & Format Trajectory Dataset (`sft_dataset.jsonl`)

In [ ]:
import os
from datasets import load_dataset

dataset_path = "/content/sft_dataset.jsonl"
if not os.path.exists(dataset_path):
    dataset_path = "/kaggle/input/aira-sft-dataset/sft_dataset.jsonl"
if not os.path.exists(dataset_path):
    dataset_path = "/kaggle/input/sft-dataset/sft_dataset.jsonl"
if not os.path.exists(dataset_path):
    dataset_path = "sft_dataset.jsonl"

print(f"[*] Loading trajectory dataset from {dataset_path}...")
dataset = load_dataset("json", data_files=dataset_path, split="train")

def format_prompts(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        # Enforce max_seq_length token limit inside the dataset itself to completely bypass SFTTrainer config changes
        tokens = tokenizer.encode(text, add_special_tokens=False)[:max_seq_length]
        texts.append(tokenizer.decode(tokens))
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True, remove_columns=dataset.column_names)
print(f"[SUCCESS] Dataset loaded! Formatted {len(dataset)} SFT samples. Columns: {dataset.column_names}")

### 5. Training Configuration (TRL SFTTrainer)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,          # Use processing_class to resolve newest TRL requirements
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,   # Cut VRAM peak in half
        gradient_accumulation_steps = 8,   # Maintain effective batch size of 8
        warmup_steps = 5,
        max_steps = 647,                   # Configure a full 1-epoch training run over the 5,176 samples
        learning_rate = 2e-4,
        fp16 = False,
        bf16 = False,
        gradient_checkpointing = True,
        logging_steps = 1,
        optim = "paged_adamw_8bit",        # Paged 8-bit optimizer to save 1.2 GB VRAM
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        save_strategy = "no"
    ),
)

In [ ]:
trainer_stats = trainer.train()

### 7. Export weights to 4-bit GGUF (for local Ollama deployment)

In [ ]:
# Save the fine-tuned LoRA adapter weights in PyTorch format to bypass Safetensors copy limitations
model.save_pretrained("gemma-4-e4b-aira-lora", safe_serialization=False)
tokenizer.save_pretrained("gemma-4-e4b-aira-lora")

# To deploy locally via Ollama:
# 1. Download your LoRA adapter folder from Colab.
# 2. Use a tool like llama.cpp to merge and quantize the model, or push the adapter directly to Hugging Face.

In [ ]:
# ============================================================
# AIRA: Base vs Fine-Tuned Evaluation
# Run this AFTER finetune_gemma.ipynb cells complete
# ============================================================

import json
import time
import re
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# ── Config ──────────────────────────────────────────────────
BASE_MODEL_ID   = "google/gemma-4-e4b-it"
ADAPTER_PATH    = "./gemma-4-e4b-aira-lora"   # where finetune saved the adapter
DATASET_PATH    = "./sft_dataset.jsonl"        # upload this to Colab
TEST_SAMPLE_N   = 100                          # held-out samples to evaluate
MAX_NEW_TOKENS  = 512
SEED            = 42

# OPA blast radius limit (must match opa_engine.py)
BLAST_RADIUS_LIMIT = 0.75

# ── Valid schema fields ──────────────────────────────────────
REQUIRED_FIELDS = {"action_type", "target_resource", "reasoning", "blast_radius"}
VALID_ACTION_TYPES = {
    "cve_exploit", "secret_exfiltration", "rbac_escalation",
    "network_pivot", "privilege_escalation",
    "image_update", "secret_rotation", "rbac_patch",
    "network_policy", "pod_restart"
}

# ── Load held-out test set ───────────────────────────────────
print("[*] Loading dataset...")
all_samples = []
with open(DATASET_PATH) as f:
    for line in f:
        all_samples.append(json.loads(line.strip()))

# Reproducible shuffle, take last 10% as test set
import random
random.seed(SEED)
random.shuffle(all_samples)
test_samples = all_samples[-TEST_SAMPLE_N:]
print(f"[+] Test set: {len(test_samples)} samples")

# ── Extract prompt from ChatML format ────────────────────────
def extract_prompt(sample):
    """Pull the user turn from the messages list."""
    msgs = sample.get("messages", [])
    system_msg = next((m["content"] for m in msgs if m["role"] == "system"), "")
    user_msg   = next((m["content"] for m in msgs if m["role"] == "user"),   "")
    return system_msg, user_msg

def extract_assistant_ground_truth(sample):
    msgs = sample.get("messages", [])
    return next((m["content"] for m in msgs if m["role"] == "assistant"), "")

# ── Metrics ──────────────────────────────────────────────────
def parse_output(text):
    """Try to extract JSON from model output. Returns dict or None."""
    # Try to find JSON block
    match = re.search(r'\{[\s\S]*\}', text)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

def evaluate_output(raw_output):
    """Returns a dict of boolean metrics for one model response."""
    results = {
        "has_thought_block":    bool(re.search(r'<thought>[\s\S]*?</thought>', raw_output, re.IGNORECASE)),
        "parseable_json":       False,
        "schema_adherent":      False,
        "valid_action_type":    False,
        "opa_compliant":        False,
    }

    parsed = parse_output(raw_output)
    if parsed is None:
        return results

    results["parseable_json"] = True

    # Schema check
    if REQUIRED_FIELDS.issubset(parsed.keys()):
        results["schema_adherent"] = True

    # Action type check
    if parsed.get("action_type") in VALID_ACTION_TYPES:
        results["valid_action_type"] = True

    # OPA check: blast_radius must be numeric and <= limit
    br = parsed.get("blast_radius")
    if isinstance(br, (int, float)) and br <= BLAST_RADIUS_LIMIT:
        results["opa_compliant"] = True

    return results

# ── Run inference for one model ──────────────────────────────
def run_eval(model, tokenizer, samples, model_label):
    print(f"\n[*] Evaluating: {model_label}")
    all_metrics = []
    latencies   = []

    for i, sample in enumerate(samples):
        system_msg, user_msg = extract_prompt(sample)

        # Format as ChatML
        messages = [
            {"role": "user", "content": f"{system_msg}\n\n{user_msg}"}
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        t0 = time.time()
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,          # greedy for reproducibility
                pad_token_id=tokenizer.eos_token_id
            )
        latency = time.time() - t0
        latencies.append(latency)

        # Decode only the new tokens
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

        metrics = evaluate_output(raw_output)
        all_metrics.append(metrics)

        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{len(samples)}] parseable={metrics['parseable_json']}, "
                  f"schema={metrics['schema_adherent']}, opa={metrics['opa_compliant']}")

    # Aggregate
    n = len(all_metrics)
    summary = {
        "model":               model_label,
        "n_samples":           n,
        "thought_block_rate":  sum(m["has_thought_block"]  for m in all_metrics) / n * 100,
        "parseable_json_rate": sum(m["parseable_json"]     for m in all_metrics) / n * 100,
        "schema_adherence":    sum(m["schema_adherent"]    for m in all_metrics) / n * 100,
        "valid_action_type":   sum(m["valid_action_type"]  for m in all_metrics) / n * 100,
        "opa_compliance":      sum(m["opa_compliant"]      for m in all_metrics) / n * 100,
        "avg_latency_sec":     sum(latencies) / len(latencies),
    }
    return summary

# ── Load base model ──────────────────────────────────────────
print("\n[*] Loading BASE model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
base_results = run_eval(base_model, tokenizer, test_samples, "Base gemma-4-e4b-it")

# ── Load fine-tuned model (base + adapter) ───────────────────
print("\n[*] Loading FINE-TUNED model (base + LoRA adapter)...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_results = run_eval(ft_model, tokenizer, test_samples, "Fine-tuned gemma-4-e4b-aira-lora")

# ── Print results table ──────────────────────────────────────
print("\n" + "="*60)
print("  EVALUATION RESULTS")
print("="*60)

metrics_display = [
    ("Thought Block Rate",    "thought_block_rate"),
    ("Parseable JSON Rate",   "parseable_json_rate"),
    ("Schema Adherence",      "schema_adherence"),
    ("Valid Action Type",     "valid_action_type"),
    ("OPA Governance Compliance", "opa_compliance"),
    ("Avg Inference Latency (s)", "avg_latency_sec"),
]

header = f"{'Metric':<30} {'Base':>12} {'Fine-Tuned':>12}"
print(header)
print("-" * len(header))
for label, key in metrics_display:
    b_val = base_results[key]
    f_val = ft_results[key]
    if key == "avg_latency_sec":
        print(f"{label:<30} {b_val:>11.2f}s {f_val:>11.2f}s")
    else:
        print(f"{label:<30} {b_val:>11.1f}% {f_val:>11.1f}%")

# ── Save to JSON ─────────────────────────────────────────────
output = {"base": base_results, "finetuned": ft_results}
with open("eval_report_real.json", "w") as f:
    json.dump(output, f, indent=2)
print("\n[+] Results saved to eval_report_real.json")
print("    Download this file and commit it to docs/")
